# MRP / 03 — Test window 2023-07-01 – 2025-12-31 (one look): gate T and gate E

Protocol: [`MRP 001`](../../discussions/MNQ_Return_Prediction/001_claude_mrp01_preregistration.md) §7–§8. **Run once, after `02/` is committed.** It refuses to run without `02/summary.json`, and if gate S failed it refuses unless `RUN_ANYWAY = True` (the test window is then spent on a model that already failed).

- Same procedure, yearly refits at 2023-07-01, 2024-01-01, 2025-01-01, fitted on everything since 2020-01-01.
- **T**: the model chosen in 02 keeps R²_OOS > 0 vs M0 with Clark–West p < 0.05 and Brier below 0.5's.
- **E** (only if S passed): one MNQ contract in the sign of μ̂ when the expected move exceeds 1 tick + 2 commissions, held one bar; net P&L > 0 with a day-block bootstrap 95% lower bound > 0.

Set `COMMISSION_PER_SIDE` to your Lucid MNQ rate and `COMMISSION_CONFIRMED = True` before running.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("MRP_SYNTHETIC") == "1"
if SYNTHETIC:                                   # never touch the real market-data folder in a smoke run
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("MRP_SYNTH_DIR") or tempfile.mkdtemp(prefix="mrp_")
    os.environ["MRP_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import mnq_predict as P

CFG = P.SYNTH_CFG if SYNTHETIC else P.MRPConfig()
OUT = Path("03"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("engine", P.MRP_VERSION, "| synthetic" if SYNTHETIC else "| real data", "| train", CFG.train)

if SYNTHETIC and not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
    P.write_synthetic(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.test[1], CFG)


In [ ]:
COMMISSION_PER_SIDE = 0.50        # USD per contract per side (Lucid) -- confirm
COMMISSION_CONFIRMED = SYNTHETIC   # set True after confirming the rate
RUN_ANYWAY = False

S = json.loads(Path("02/summary.json").read_text())
if not COMMISSION_CONFIRMED:
    raise RuntimeError("Confirm the Lucid commission and set COMMISSION_CONFIRMED = True.")
if not S["gate_S"]["pass"] and not RUN_ANYWAY:
    raise RuntimeError("Gate S failed in 02: the test window is not opened (set RUN_ANYWAY = True to override).")
from dataclasses import replace
CFG = replace(CFG, commission_per_side=COMMISSION_PER_SIDE)
SCALE, BEST = S["scale"], S["gate_S"]["best_model"]
print("scale", SCALE, "| model", BEST, "| commission", COMMISSION_PER_SIDE)

In [ ]:
grid, fx = P.load_inputs(CFG.test[1], CFG)
dss = {r: P.dataset_for(grid, fx, CFG.train[0], r, CFG, SCALE) for r in CFG.refits_test}
fc = P.walk_forward(dss, CFG.refits_test, CFG.test[1], CFG.train[0], CFG)
td = fc["trade_date"].to_numpy()
periods = {"all": np.ones(len(fc), bool)}
for yv in sorted(set(pd.to_datetime(td).year)):
    periods[str(yv)] = pd.to_datetime(td).year == yv
rep = P.forecast_report(fc, periods)
flat = pd.DataFrame({k: {m: (v["p_one_sided"] if isinstance(v, dict) else v) for m, v in r.items()} for k, r in rep.items()})
display(flat)

In [ ]:
r2 = rep["all"][f"r2_{BEST}_vs_M0"]; cwp = rep["all"][f"cw_{BEST}_vs_M0"]["p_one_sided"]
T = bool(r2 > 0 and cwp < 0.05 and rep["all"][f"brier_{BEST}"] < rep["all"]["brier_half"])
print(f"Gate T ({BEST}): R2_OOS {r2:.5f}, CW p {cwp:.4f}, Brier {rep['all'][f'brier_{BEST}']:.5f} vs 0.25 -> {'PASS' if T else 'FAIL'}")
cal = P.calibration(fc[f"pup_{BEST}"].to_numpy(), fc["ret"].to_numpy()); display(cal.round(4))
cover = P.interval_coverage(fc, BEST); print("interval coverage", cover)

## Gate E (one-bar trade, net of costs)

In [ ]:
E = None
if S["gate_S"]["pass"]:
    tr = P.one_bar_pnl(fc, BEST, CFG, SCALE)
    daily = tr.groupby("trade_date")["net_usd"].sum()
    bs = P.day_block_bootstrap(daily.to_numpy())
    n_tr = int((tr["pos"] != 0).sum())
    E = bool(bs["total"] > 0 and bs["lower_95"] > 0)
    print(f"trades {n_tr}, gross ${tr['gross_usd'].sum():,.0f}, net ${tr['net_usd'].sum():,.0f}, "
          f"bootstrap 95% lower ${bs['lower_95']:,.0f} -> E {'PASS' if E else 'FAIL'}")
    ax = daily.cumsum().plot(figsize=(11, 3.5), title=f"cumulative net P&L, 1 MNQ contract ({BEST})")
    plt.tight_layout(); plt.savefig(OUT / "pnl.png", dpi=110); plt.show()
else:
    print("Gate E not applicable: S failed.")
summary = {"engine": P.MRP_VERSION, "synthetic": SYNTHETIC, "scale": SCALE, "model": BEST, "forecast": rep,
           "coverage": cover, "gate_T": T, "gate_E": E, "commission_per_side": COMMISSION_PER_SIDE}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1, default=lambda o: bool(o) if isinstance(o, np.bool_) else float(o)))